In [12]:
# main.py
import os
from utils.data_preprocessing import fq_pie_pre_process_extract, dualpi2_pre_process_extract, trim_df
from utils.utils import get_stats, find_files_with_extension, extract_unique_mbps_and_ms
import pandas as pd

In [13]:
columns_to_use = [
    'queue_type', 'burst_allowance', 'drop_probability', 'current_queue_delay',
    'accumulated_probability', 'average_dequeue_time', 'length_in_bytes', 'total_drops', 'packet_length'
]

In [14]:
folderpaths = ['../../data/udp_data_2025-04-27/kernel_data','../../data/udp_data_2025-04-28/kernel_data']

In [15]:
filenames, filepaths, filedict = find_files_with_extension(paths=folderpaths, extension='.txt')

In [16]:
# Call the function and print unique combinations
unique_combinations = extract_unique_mbps_and_ms(folderpaths)
print("Unique Mbps and ms Combinations:")
for mbps, ms in sorted(unique_combinations):
    print(f'{mbps} Mbps, {ms} ms')

unique_scenarios_dict_v2 = {}
for mbps, ms in unique_combinations:
    scenario_search = f"{mbps}Mbps_{ms}ms"
    for index in range(len(filenames)):
        if scenario_search in filenames[index]:
            print(f"Found {scenario_search} in {filenames[index]}")
            if scenario_search in unique_scenarios_dict_v2:
                unique_scenarios_dict_v2[scenario_search].append(filedict[filenames[index]])
            else:
                unique_scenarios_dict_v2[scenario_search]=[filedict[filenames[index]]]
            


Unique Mbps and ms Combinations:
10 Mbps, 0 ms
10 Mbps, 10 ms
10 Mbps, 20 ms
10 Mbps, 5 ms
100 Mbps, 0 ms
100 Mbps, 10 ms
100 Mbps, 20 ms
100 Mbps, 5 ms
150 Mbps, 0 ms
150 Mbps, 10 ms
150 Mbps, 20 ms
150 Mbps, 5 ms
20 Mbps, 0 ms
20 Mbps, 10 ms
20 Mbps, 20 ms
20 Mbps, 5 ms
200 Mbps, 0 ms
200 Mbps, 10 ms
200 Mbps, 20 ms
200 Mbps, 5 ms
250 Mbps, 0 ms
250 Mbps, 10 ms
250 Mbps, 20 ms
250 Mbps, 5 ms
5 Mbps, 0 ms
5 Mbps, 10 ms
5 Mbps, 20 ms
5 Mbps, 5 ms
Found 20Mbps_20ms in kernel_data_1_dualpi2_20Mbps_20ms_ecn_udp.txt
Found 20Mbps_20ms in kernel_data_2_dualpi2_20Mbps_20ms_ecn_udp.txt
Found 100Mbps_10ms in kernel_data_1_dualpi2_100Mbps_10ms_ecn_udp.txt
Found 100Mbps_10ms in kernel_data_2_dualpi2_100Mbps_10ms_ecn_udp.txt
Found 10Mbps_10ms in kernel_data_1_dualpi2_10Mbps_10ms_ecn_udp.txt
Found 10Mbps_10ms in kernel_data_2_dualpi2_10Mbps_10ms_ecn_udp.txt
Found 250Mbps_0ms in kernel_data_1_dualpi2_250Mbps_0ms_ecn_udp.txt
Found 250Mbps_0ms in kernel_data_2_dualpi2_250Mbps_0ms_ecn_udp.txt
Found 150

In [17]:
# dualpi2_df['queue_type'].value_counts().sort_index(ascending=True).plot(kind='bar', title='queue_type')
# dualpi2_df['burst_allowance'].value_counts().sort_index(ascending=True).plot(kind='bar', title='Burst Allowance')

In [ ]:
import matplotlib.pyplot as plt

for scenario in unique_scenarios_dict_v2.keys():
    filepath = unique_scenarios_dict_v2[scenario][0]
    
    # Preprocess and extract data from the file
    dualpi2_df = dualpi2_pre_process_extract(input_file=filepath, aqm='dualpi2')
    
    # Plot 'current_queue_delay' vs index
    plt.figure(figsize=(10, 5))
    plt.plot(dualpi2_df.index, dualpi2_df['current_queue_delay'], label=scenario)
    plt.title(f'Queue Delay for Scenario: {scenario}')
    plt.xlabel('Time')
    plt.ylabel('Current Queue Delay')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
# Initialize an empty list to store the results
results = []

for scenario in unique_scenarios_dict_v2.keys():
    filepath = unique_scenarios_dict_v2[scenario][0]
    
    # Preprocess and extract data from the file
    dualpi2_df = dualpi2_pre_process_extract(input_file=filepath, aqm='dualpi2')
    
    # Calculate the means for the different queue types
    overallqd_mean = dualpi2_df['current_queue_delay'].mean()
    cqd_mean = dualpi2_df[dualpi2_df['queue_type'] == 0]['current_queue_delay'].mean()
    lqd_mean = dualpi2_df[dualpi2_df['queue_type'] == 1]['current_queue_delay'].mean()

    # Extract the value counts for dequeue_action and queue_type as strings
    dequeue_action_counts = dualpi2_df['dequeue_action'].value_counts().to_dict()
    queue_type_counts = dualpi2_df['queue_type'].value_counts().to_dict()
    
    # Convert the value counts to strings to easily add to the dataframe
    dequeue_action_str = str(dequeue_action_counts)
    queue_type_str = str(queue_type_counts)

    # Append the means, value counts, and filepath to the results list
    results.append({
        'scenario': scenario,
        'overallqd_mean': overallqd_mean,
        'cqd_mean': cqd_mean,
        'lqd_mean': lqd_mean,
        'dequeue_action_counts': dequeue_action_str,
        'queue_type_counts': queue_type_str
    })

# Convert the results list to a DataFrame
results_df = pd.DataFrame(results)

# Print the dataframe
print(results_df)

# Convert mean values from ms to seconds (divide by 1000)
results_df['overallqd_mean'] = results_df['overallqd_mean'] / 1000
results_df['cqd_mean'] = results_df['cqd_mean'] / 1000
results_df['lqd_mean'] = results_df['lqd_mean'] / 1000

# Function to extract bandwidth and delay from the scenario column
def extract_bandwidth_delay(scenario):
    bandwidth, delay = scenario.split('_')
    bandwidth_value = int(bandwidth.replace('Mbps', ''))
    delay_value = int(delay.replace('ms', ''))
    return bandwidth_value, delay_value

# Extract bandwidth and delay values
results_df[['bandwidth', 'delay']] = results_df['scenario'].apply(extract_bandwidth_delay).apply(pd.Series)

# Sort first by bandwidth, then by delay
df_sorted = results_df.sort_values(by=['bandwidth', 'delay'])


# Save the sorted dataframe to a CSV file
df_sorted.to_csv('kernel_queue_delay_results.csv', index=False)

df_sorted = results_df.sort_values(by=['overallqd_mean'])
df_sorted.to_csv('kernel_queue_delay_results_sorted_ascending.csv', index=False)


        scenario  overallqd_mean      cqd_mean      lqd_mean  \
0    20Mbps_20ms    29190.206953  38615.370194  25720.026032   
1   100Mbps_10ms    13189.980355  17969.107796   8299.333136   
2    10Mbps_10ms    32235.461033  39364.791758  29562.394456   
3    250Mbps_0ms      147.252771    260.665845     61.247008   
4   150Mbps_20ms     5828.018269   6756.691909   3284.545074   
5    20Mbps_10ms    35306.035272  42504.326712  32535.333801   
6      5Mbps_5ms    40703.355367  49955.810646  37191.385992   
7    250Mbps_5ms     1025.808960   1340.372625    307.526073   
8   200Mbps_20ms     2698.568775   3187.913209   1081.979693   
9   150Mbps_10ms    13420.453068  16301.993474   7300.852243   
10    5Mbps_20ms    36411.466404  50610.726734  31138.756351   
11  200Mbps_10ms     5502.261966   6133.400336   2947.067585   
12    5Mbps_10ms    40097.522699  52194.872320  35536.303471   
13  250Mbps_20ms     1675.641497   1981.796089    559.284384   
14   100Mbps_0ms     6521.922730   8901.

In [ ]:
for filepath in filepaths:
    dualpi2_df = dualpi2_pre_process_extract(input_file=filepath,aqm='dualpi2')
    overallqd_mean = dualpi2_df['current_queue_delay'].mean()
    cqd_mean = dualpi2_df[dualpi2_df['queue_type'] == 0]['current_queue_delay'].mean()
    lqd_mean = dualpi2_df[dualpi2_df['queue_type'] == 1]['current_queue_delay'].mean()

    if cqd_mean < lqd_mean:
        print("L4S Queue Delay is greater than Classic Queue Delay")
    
        
        print()
        print("*** Start ***"*3)
        print(filepath)
        print("dequeu_actions",dualpi2_df['dequeue_action'].value_counts())
        print("queue_type",dualpi2_df['queue_type'].value_counts())
        print("Classic Queue: ",dualpi2_df[dualpi2_df['queue_type'] == 0]['current_queue_delay'].mean())
        print("L4S Queue: ",dualpi2_df[dualpi2_df['queue_type'] == 1]['current_queue_delay'].mean())

    


In [ ]:
for filepath in filepaths:
    
    dualpi2_df = dualpi2_pre_process_extract(input_file=filepath,aqm='dualpi2')

    if dualpi2_df['current_queue_delay'].mean() < 10000:
        print()
        print("*** Start ***"*3)
        print(filepath)
        print("dequeu_actions",dualpi2_df['dequeue_action'].value_counts())
        print("queue_type",dualpi2_df['queue_type'].value_counts())
        print("Classic Queue Stats")
        get_stats(dualpi2_df[dualpi2_df['queue_type'] == 0], columns_to_use[3])
        print("L4S Queue Stats")
        get_stats(dualpi2_df[dualpi2_df['queue_type'] == 1], columns_to_use[3])
        # get_stats(dualpi2_df, columns_to_use[3])
        print()
        print("*** END ***"*3)
    


In [ ]:
for filepath in filepaths:
    
    dualpi2_df = dualpi2_pre_process_extract(input_file=filepath,aqm='dualpi2')

    if dualpi2_df['current_queue_delay'].mean() < 10000 and len(dualpi2_df['dequeue_action'].unique()) > 1:
        print()
        print("*** Start ***"*3)
        print(filepath)
        print("dequeu_actions",dualpi2_df['dequeue_action'].value_counts())
        print("queue_type",dualpi2_df['queue_type'].value_counts())
        get_stats(dualpi2_df[dualpi2_df['queue_type'] == 0], columns_to_use[3])
        get_stats(dualpi2_df[dualpi2_df['queue_type'] == 1], columns_to_use[3])
        # get_stats(dualpi2_df, columns_to_use[3])
        print()
        print("*** END ***"*3)
    
